# 03 · Bronze → Silver (validate + enrich)

- **users** — flag valid emails, build `full_name`
- **items** — clamp negative prices, upper-case `category`
- **purchases_enriched** — join users + items, derive `total_price`, `purchase_date`, `purchase_hour`
- **pageviews_by_items** — parse `/products/{id}` URLs, join items

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("03-bronze-to-silver").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| default catalog:", spark.conf.get("spark.sql.defaultCatalog"))

In [ ]:
from pyspark.sql.functions import (
    col, concat_ws, hour, lit, regexp_extract, to_date, upper, when,
)

users = spark.table("demo.bronze.users")
items = spark.table("demo.bronze.items")
purchases = spark.table("demo.bronze.purchases")
pageviews = spark.table("demo.bronze.pageviews")

def overwrite(df, table):
    df.write.format("iceberg").mode("overwrite").save(table)
    print(table, "->", df.count(), "rows")

## silver.users — email validation + full name

In [ ]:
EMAIL_RE = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
silver_users = (
    users.withColumn("valid_email", col("email").rlike(EMAIL_RE))
         .withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name")))
         .select("id", "first_name", "last_name", "email",
                 "created_at", "updated_at", "valid_email", "full_name")
)
silver_users.select("full_name", "email", "valid_email").show(5, truncate=False)
overwrite(silver_users, "demo.silver.users")

## silver.items — clamp price, normalise category

In [ ]:
silver_items = (
    items.withColumn("price", when(col("price") < 0, lit(0)).otherwise(col("price")))
         .withColumn("category", upper(col("category")))
         .select("id", "name", "category", "price", "inventory", "created_at", "updated_at")
)
overwrite(silver_items, "demo.silver.items")

## silver.purchases_enriched — join + derive

In [ ]:
enriched = (
    purchases.alias("p")
    .join(users.alias("u"), col("p.user_id") == col("u.id"), "left")
    .join(items.alias("i"), col("p.item_id") == col("i.id"), "left")
    .select(
        col("p.id"), col("p.user_id"), col("p.item_id"), col("p.quantity"),
        col("p.purchase_price"),
        (col("p.quantity") * col("p.purchase_price")).cast("decimal(14,2)").alias("total_price"),
        col("u.email").alias("user_email"),
        col("i.name").alias("item_name"),
        col("i.category").alias("item_category"),
        to_date(col("p.created_at")).alias("purchase_date"),
        hour(col("p.created_at")).alias("purchase_hour"),
        col("p.created_at"), col("p.updated_at"),
    )
)
enriched.select("item_name", "quantity", "purchase_price", "total_price", "purchase_date").show(5, truncate=False)
overwrite(enriched, "demo.silver.purchases_enriched")

## silver.pageviews_by_items — parse URL + join items

In [ ]:
parsed = (
    pageviews.withColumn("page", regexp_extract(col("url"), r"^/([^/]+)/\d+$", 1))
             .withColumn("item_id", regexp_extract(col("url"), r"/(\d+)$", 1).cast("bigint"))
             .filter(col("item_id").isNotNull())
)
pv_by_items = (
    parsed.alias("v")
    .join(items.alias("i"), col("v.item_id") == col("i.id"), "left")
    .select(
        col("v.user_id"), col("v.item_id"), col("v.page"),
        col("i.name").alias("item_name"), col("i.category").alias("item_category"),
        col("v.channel"), col("v.received_at"),
    )
)
pv_by_items.show(5, truncate=False)
overwrite(pv_by_items, "demo.silver.pageviews_by_items")